In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error

# ---------- 1. Load data ----------
path = '/kaggle/input/competitions/store-sales-time-series-forecasting/'
train = pd.read_csv(path + 'train.csv', parse_dates=['date'])
test  = pd.read_csv(path + 'test.csv', parse_dates=['date'])
stores = pd.read_csv(path + 'stores.csv')
oil = pd.read_csv(path + 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(path + 'holidays_events.csv', parse_dates=['date'])

# ---------- 2. Fix oil.csv (fill gaps) ----------
full_dates = pd.DataFrame({'date': pd.date_range(train.date.min(), test.date.max())})
oil = full_dates.merge(oil, on='date', how='left')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

# ---------- 3. Fix holidays (handle transferred logic) ----------
actual_holidays = holidays[
    ((holidays['type'] == 'Holiday') & (holidays['transferred'] == False)) |
    (holidays['type'].isin(['Transfer', 'Bridge', 'Additional', 'Event']))
].copy()

national = actual_holidays[actual_holidays['locale'] == 'National'][['date']].drop_duplicates()
national['is_national_holiday'] = 1

regional = actual_holidays[actual_holidays['locale'] == 'Regional'][['date','locale_name']].drop_duplicates()
regional = regional.rename(columns={'locale_name':'state'})
regional['is_regional_holiday'] = 1

local = actual_holidays[actual_holidays['locale'] == 'Local'][['date','locale_name']].drop_duplicates()
local = local.rename(columns={'locale_name':'city'})
local['is_local_holiday'] = 1

# ---------- 4. Merge everything ----------
def merge_all(df):
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    df = df.merge(national, on='date', how='left')
    df = df.merge(regional, on=['date','state'], how='left')
    df = df.merge(local, on=['date','city'], how='left')
    for col in ['is_national_holiday','is_regional_holiday','is_local_holiday']:
        df[col] = df[col].fillna(0).astype(int)
    df['is_holiday'] = ((df['is_national_holiday'] + df['is_regional_holiday'] + df['is_local_holiday']) > 0).astype(int)
    return df

train = merge_all(train)
test = merge_all(test)

# ---------- 5. Date features ----------
def add_date_features(df):
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_payday'] = ((df['day'] == 15) | (df['date'].dt.is_month_end)).astype(int)
    return df

train = add_date_features(train)
test = add_date_features(test)

# ---------- 6. Encode categoricals (XGBoost needs numeric codes, not pandas 'category' dtype with native handling like LGBM) ----------
cat_cols = ['family', 'city', 'state', 'type']
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category').cat.set_categories(train[col].cat.categories)
    train[col] = train[col].cat.codes
    test[col] = test[col].cat.codes

# ---------- 7. Target transform (RMSLE-friendly) ----------
train['sales'] = train['sales'].clip(lower=0)
train['log_sales'] = np.log1p(train['sales'])

# ---------- 8. Train/validation split (time-based) ----------
cutoff = train['date'].max() - pd.Timedelta(days=15)
tr = train[train['date'] <= cutoff]
val = train[train['date'] > cutoff]

features = ['store_nbr','family','onpromotion','city','state','type','cluster',
            'dcoilwtico','is_holiday','year','month','day','dayofweek',
            'is_weekend','is_payday']

dtrain = xgb.DMatrix(tr[features], label=tr['log_sales'])
dval = xgb.DMatrix(val[features], label=val['log_sales'])

# ---------- 9. Train model ----------
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbosity': 0
}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=50
)

# ---------- 10. Validate (RMSLE) ----------
val_preds = np.expm1(model.predict(dval, iteration_range=(0, model.best_iteration + 1)))
val_preds = np.clip(val_preds, 0, None)
rmsle = np.sqrt(mean_squared_log_error(val['sales'], val_preds))
print(f'Validation RMSLE: {rmsle:.5f}')

# ---------- 11. Retrain on FULL data, then predict test ----------
dfull = xgb.DMatrix(train[features], label=train['log_sales'])
dtest = xgb.DMatrix(test[features])

final_model = xgb.train(params, dfull, num_boost_round=model.best_iteration + 1)
test_preds = np.expm1(final_model.predict(dtest))
test_preds = np.clip(test_preds, 0, None)

# ---------- 12. Submission ----------
submission = pd.DataFrame({'id': test['id'], 'sales': test_preds})
submission.to_csv('submission.csv', index=False)
print(submission.head())

[0]	val-rmse:2.51075
[50]	val-rmse:0.74033
[100]	val-rmse:0.61635
[150]	val-rmse:0.56645
[200]	val-rmse:0.54281
[250]	val-rmse:0.52398
[300]	val-rmse:0.51560
[350]	val-rmse:0.50911
[400]	val-rmse:0.50382
[450]	val-rmse:0.49703
[500]	val-rmse:0.49320
[550]	val-rmse:0.48923
[600]	val-rmse:0.48507
[650]	val-rmse:0.48187
[700]	val-rmse:0.47976
[750]	val-rmse:0.47721
[800]	val-rmse:0.47444
[850]	val-rmse:0.47172
[900]	val-rmse:0.47028
[950]	val-rmse:0.46771
[999]	val-rmse:0.46701
Validation RMSLE: 0.46471
        id        sales
0  3000888     4.046260
1  3000889     0.170543
2  3000890     5.645353
3  3000891  2488.422607
4  3000892     0.170579
